# LeetCode 863 — All Nodes Distance K in Binary Tree bằng Python

Link đề: https://leetcode.com/problems/all-nodes-distance-k-in-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, DFS, BFS, Hash Table, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def distanceK(self, root: TreeNode, target: TreeNode, k: int) -> List[int]:
        ...
```

Ghi chú: kết quả là **tập giá trị, thứ tự bất kỳ**, nên test so sánh sau khi `sorted`. Trên LeetCode `target` thực ra là **nút** (`TreeNode`), dù ví dụ đề ghi theo giá trị; notebook tự dựng `TreeNode`, hàm dựng cây kiểu LeetCode (`build_tree` từ danh sách theo tầng với `None`) và hàm `find` đổi giá trị → nút. Oracle: **Floyd–Warshall** trên đồ thị vô hướng của cây (vét cạn mọi hình dạng cây nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm mọi nút cách nút `target` đúng `k` cạnh (đi lên cha hoặc xuống con đều tính).
2. Nắm công cụ Python: lớp `TreeNode`, dựng cây từ danh sách theo tầng, `dict` khoá bằng **nút** (băm theo danh tính), `deque`, tập `seen`, đệ quy kèm tham số "đến từ đâu".
3. Đi từ **tính khoảng cách mọi nút bằng đường đi từ gốc (brute force)** → **bản đồ cha + BFS từ target** → **bản đồ cha + DFS bỏ nút vừa đến** → **một lượt DFS trả khoảng cách tới target (không cần bản đồ cha)**.
4. Hiểu mấu chốt: cây là đồ thị vô hướng nhưng con trỏ chỉ đi xuống; muốn đi lên phải **biết cha** (bản đồ cha) hoặc **để đệ quy báo khoảng cách ngược lên**.
5. Tránh lỗi kinh điển: chỉ đi xuống, quay lại nút vừa đến (đếm trùng / lặp vô hạn), quên nhánh anh em khi đi lên, nhầm `k` (lệch một), `k = 0` phải trả về chính `target`, và so sánh nút theo giá trị thay vì danh tính.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân, nút `target` và số nguyên `k`. Trả về mảng giá trị của **mọi** nút có khoảng cách đúng `k` tới `target` (khoảng cách là số cạnh trên đường đi ngắn nhất, được đi cả lên và xuống). Thứ tự tuỳ ý.

**Ví dụ 1:** `root = [3,5,1,6,2,0,8,null,null,7,4]`, `target = 5`, `k = 2` → `[7,4,1]`.

**Ví dụ 2:** `root = [1]`, `target = 1`, `k = 3` → `[]`.

**Ràng buộc:** số nút `[1, 500]`; `0 <= Node.val <= 500`; mọi giá trị **duy nhất**; `target` là một nút trong cây; `0 <= k <= 1000`.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, x=0, left=None, right=None):
        self.val = x
        self.left = left
        self.right = right


def build_tree(vals):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None là nút trống)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def to_level(root):
    """Ngược lại: danh sách theo tầng, cắt None ở cuối."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def all_nodes(root):
    """Mọi nút (tiền thứ tự, lặp, không đệ quy)."""
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node)
        if node.right:
            st.append(node.right)
        if node.left:
            st.append(node.left)
    return out


def find(root, val):
    """Đổi giá trị -> nút (đề cho target theo giá trị, LeetCode truyền nút)."""
    for node in all_nodes(root):
        if node.val == val:
            return node
    raise KeyError(val)


ex1_list = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
ONE = build_tree([1])                                  # cây một nút (target chính là gốc)
root1 = build_tree(ex1_list)
assert to_level(root1) == ex1_list
print("cây ví dụ 1 dựng lại:", to_level(root1), "| target 5 ->", find(root1, 5).val)

cây ví dụ 1 dựng lại: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4] | target 5 -> 5


## 3. Kiến thức nền

### 3.1 Cây là đồ thị vô hướng

Khoảng cách giữa hai nút = số cạnh trên **đường đi duy nhất** giữa chúng (cây không có chu trình). Đường đi đi **lên** tới tổ tiên chung thấp nhất (LCA) rồi **xuống**. Nhưng `TreeNode` chỉ có `left`, `right`: không có con trỏ cha. Muốn đi lên phải tự tạo thông tin cha.

In [2]:
def parent_map(root):
    parent = {root: None}                               # khoá = NÚT (băm theo danh tính), không phải giá trị
    st = [root]
    while st:
        node = st.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                st.append(child)
    return parent


par = parent_map(root1)
t5 = find(root1, 5)
print("cha của 5:", par[t5].val, "| cha của 7:", par[find(root1, 7)].val, "| cha của gốc:", par[root1])

cha của 5: 3 | cha của 7: 2 | cha của gốc: None


### 3.2 Khoá `dict` bằng nút, tập `seen` bằng nút

Lớp `TreeNode` mặc định **băm theo danh tính** (`id`), nên dùng nút làm khoá `dict`/phần tử `set` là chính xác và nhanh; hai nút có cùng giá trị vẫn khác nhau. (Đề đảm bảo giá trị duy nhất, nhưng làm theo nút là thói quen an toàn.)

### 3.3 BFS theo tầng

`deque` + vòng lặp theo tầng: mỗi lần xử lý đúng `len(queue)` phần tử hiện có là một "bán kính". Sau `k` tầng, hàng đợi chính là tập nút cách nguồn đúng `k` cạnh (nếu đánh dấu đã thăm).

In [3]:
q, seen = deque([t5]), {t5}
for d in range(3):
    print(f"tầng {d}:", sorted(n.val for n in q))
    nxt = deque()
    for node in q:
        for nb_ in (node.left, node.right, par[node]):
            if nb_ and nb_ not in seen:
                seen.add(nb_)
                nxt.append(nb_)
    q = nxt

tầng 0: [5]
tầng 1: [2, 3, 6]
tầng 2: [1, 4, 7]


### 3.4 Đệ quy "đến từ đâu"

Khi đệ quy trên đồ thị vô hướng không dùng tập `seen`, ta truyền thêm nút **vừa đến** (`frm`) và bỏ qua nó: trên cây điều này đủ để không quay lại (cây không có chu trình).

### 3.5 Đệ quy trả về thông tin cho nút cha

Một kỹ thuật khác: hàm `dfs(node)` trả "khoảng cách từ `node` tới `target` nếu `target` nằm trong cây con của `node`, ngược lại `-1`". Nhờ vậy mỗi tổ tiên của `target` biết mình cách `target` bao xa mà **không cần con trỏ cha**.

## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1, `target = 5`, `k = 2`:

```
        3
      /   \
     5     1
    / \   / \
   6   2 0   8
      / \
     7   4
```

Các nút cách `5` đúng 2 cạnh:

- **xuống:** `7` và `4` (qua `2`);
- **lên rồi xuống:** `1` (qua cha `3`, rồi sang con kia).

Không có nút nào ở nhánh `6`: `6` chỉ cách `5` đúng 1 cạnh. Nhớ rằng đi lên thì còn có thể **rẽ sang nhánh anh em** của tổ tiên.

In [4]:
for d, level in enumerate([[5], [6, 2, 3], [7, 4, 1]]):
    print(f"cách target 5 đúng {d} cạnh:", level)

cách target 5 đúng 0 cạnh: [5]
cách target 5 đúng 1 cạnh: [6, 2, 3]
cách target 5 đúng 2 cạnh: [7, 4, 1]


## 5. Cách 1 — Brute Force: khoảng cách qua đường đi từ gốc

**Ý tưởng:** với mỗi nút, lưu **đường từ gốc tới nút** (danh sách các nút tổ tiên). Khoảng cách giữa `u` và `v` là `|path_u| + |path_v| − 2·|tiền tố chung|`. Duyệt mọi nút, giữ những nút có khoảng cách `k`.

In [5]:
def root_paths(root):
    paths = {root: [root]}
    st = [root]
    while st:
        node = st.pop()
        for child in (node.left, node.right):
            if child:
                paths[child] = paths[node] + [child]               # đường từ gốc tới child
                st.append(child)
    return paths


def distance_k_brute(root, target, k):
    paths = root_paths(root)
    pt = paths[target]
    res = []
    for node, pn in paths.items():
        common = 0
        while common < min(len(pt), len(pn)) and pt[common] is pn[common]:
            common += 1                                            # độ dài tiền tố chung (tới LCA)
        dist = (len(pt) - common) + (len(pn) - common)
        if dist == k:
            res.append(node.val)
    return res


print(sorted(distance_k_brute(root1, find(root1, 5), 2)), distance_k_brute(ONE, ONE, 3))
assert sorted(distance_k_brute(root1, find(root1, 5), 2)) == [1, 4, 7]

[1, 4, 7] []


**Độ phức tạp:** mỗi đường dài tới `h`, so từng cặp `O(h)` → `O(n·h)` thời gian, `O(n·h)` bộ nhớ lưu các đường (đường sao chép). Với cây lệch (`h = n = 500`) vẫn chạy, nhưng lãng phí.

**Nút thắt:** ta so từng nút với `target` từ đầu thay vì **lan toả từ `target`** theo từng cạnh. Nếu có cha, BFS từ `target` ra `k` tầng là xong.

## 6. Ý tưởng thuật toán / bước đột phá

Biến cây thành đồ thị vô hướng: mỗi nút có thêm "hàng xóm" là **cha**. Rồi:

- **BFS** từ `target`, `k` tầng; hoặc
- **DFS** từ `target` với bộ đếm còn lại, không quay lại nút vừa đến.

Hoặc không cần bản đồ cha: một lượt DFS từ gốc, mỗi nút trả "khoảng cách tới `target`"; tổ tiên của `target` ở khoảng cách `d` thu các nút ở độ sâu `k − d` trong **nhánh còn lại**.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** bản đồ cha + BFS, **Cách 3** bản đồ cha + DFS, **Cách 4** một lượt DFS không cần bản đồ cha.

### Cách 2 — Bản đồ cha + BFS

**Giải thích từng đoạn:**

1. Duyệt cây, ghi `parent[child] = node`.
2. BFS từ `target`, tập `seen` chứa nút đã thăm; mỗi tầng mở rộng sang `left`, `right`, `parent`.
3. Sau đúng `k` tầng, hàng đợi là đáp án; nếu hết nút sớm trả `[]`.

In [6]:
def distance_k_bfs(root, target, k):
    parent = parent_map(root)
    level, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in level:
            for nb_ in (node.left, node.right, parent[node]):
                if nb_ is not None and nb_ not in seen:
                    seen.add(nb_)
                    nxt.append(nb_)
        level = nxt
        if not level:
            break
    return [n.val for n in level]


print(sorted(distance_k_bfs(root1, find(root1, 5), 2)), distance_k_bfs(ONE, ONE, 3))
assert sorted(distance_k_bfs(root1, find(root1, 5), 2)) == [1, 4, 7]

[1, 4, 7] []


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionBFS:
    def distanceK(self, root, target, k):
        parent = {root: None}
        st = [root]
        while st:
            node = st.pop()
            for child in (node.left, node.right):
                if child:
                    parent[child] = node
                    st.append(child)
        level, seen = [target], {target}
        for _ in range(k):
            nxt = []
            for node in level:
                for nb in (node.left, node.right, parent[node]):
                    if nb is not None and nb not in seen:
                        seen.add(nb)
                        nxt.append(nb)
            level = nxt
            if not level:
                break
        return [n.val for n in level]

**Độ phức tạp:** `O(n)` thời gian (mỗi nút vào hàng đợi tối đa một lần), `O(n)` bộ nhớ.

### Cách 3 — Bản đồ cha + DFS bỏ nút vừa đến

**Giải thích từng đoạn:**

1. Giống Cách 2: dựng `parent`.
2. `dfs(node, frm, d)`: nếu `node` rỗng thì thoát; nếu `d == 0` thu `node.val`; ngược lại thử `left`, `right`, `parent` **trừ** `frm`, với `d − 1`.
3. Gọi `dfs(target, None, k)`.

In [8]:
def distance_k_dfs(root, target, k):
    parent = parent_map(root)
    res = []

    def dfs(node, frm, d):
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            return
        for nxt in (node.left, node.right, parent[node]):
            if nxt is not frm:                           # không quay lại nút vừa đến
                dfs(nxt, node, d - 1)

    dfs(target, None, k)
    return res


print(sorted(distance_k_dfs(root1, find(root1, 5), 2)), distance_k_dfs(ONE, ONE, 3))
assert sorted(distance_k_dfs(root1, find(root1, 5), 2)) == [1, 4, 7]

[1, 4, 7] []


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionDFS:
    def distanceK(self, root, target, k):
        parent = {root: None}
        st = [root]
        while st:
            node = st.pop()
            for child in (node.left, node.right):
                if child:
                    parent[child] = node
                    st.append(child)
        res = []

        def dfs(node, frm, d):
            if node is None:
                return
            if d == 0:
                res.append(node.val)
                return
            for nxt in (node.left, node.right, parent[node]):
                if nxt is not frm:
                    dfs(nxt, node, d - 1)

        dfs(target, None, k)
        return res

**Độ phức tạp:** `O(n)` thời gian (mỗi nút thăm tối đa một lần vì không quay lại), `O(n)` bộ nhớ (bản đồ cha + ngăn xếp đệ quy tới độ sâu `min(k, n)`).

### Cách 4 — Một lượt DFS, không cần bản đồ cha

**Ý tưởng:** `dfs(node)` trả khoảng cách `d` từ `node` tới `target` nếu `target` nằm trong cây con của `node`, ngược lại `-1`.

- Gặp `target`: thu mọi nút ở độ sâu `k` **bên dưới** nó (`down(node, k)`), trả `0`.
- Nút `node` có `target` ở cây con trái với khoảng cách `L` (nghĩa `d = L + 1` từ `node`): nếu `d == k` thu `node`; nếu `d < k` thu các nút ở độ sâu `k − d − 1` trong cây con **phải**; trả `d`. Đối xứng cho phải.

In [10]:
def distance_k_onepass(root, target, k):
    res = []

    def down(node, d):                                   # thu nút ở độ sâu d bên dưới node
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            return
        down(node.left, d - 1)
        down(node.right, d - 1)

    def dfs(node):                                       # khoảng cách node -> target, hoặc -1 nếu target không nằm dưới node
        if node is None:
            return -1
        if node is target:
            down(node, k)
            return 0
        L = dfs(node.left)
        if L != -1:
            d = L + 1
            if d == k:
                res.append(node.val)
            elif d < k:
                down(node.right, k - d - 1)              # rẽ sang nhánh anh em
            return d
        R = dfs(node.right)
        if R != -1:
            d = R + 1
            if d == k:
                res.append(node.val)
            elif d < k:
                down(node.left, k - d - 1)
            return d
        return -1

    dfs(root)
    return res


print(sorted(distance_k_onepass(root1, find(root1, 5), 2)), distance_k_onepass(ONE, ONE, 3))
assert sorted(distance_k_onepass(root1, find(root1, 5), 2)) == [1, 4, 7]

[1, 4, 7] []


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionOnePass:
    def distanceK(self, root, target, k):
        res = []

        def down(node, d):
            if node is None:
                return
            if d == 0:
                res.append(node.val)
                return
            down(node.left, d - 1)
            down(node.right, d - 1)

        def dfs(node):
            if node is None:
                return -1
            if node is target:
                down(node, k)
                return 0
            L = dfs(node.left)
            if L != -1:
                d = L + 1
                if d == k:
                    res.append(node.val)
                elif d < k:
                    down(node.right, k - d - 1)
                return d
            R = dfs(node.right)
            if R != -1:
                d = R + 1
                if d == k:
                    res.append(node.val)
                elif d < k:
                    down(node.left, k - d - 1)
                return d
            return -1

        dfs(root)
        return res

**Độ phức tạp:** `O(n)` thời gian (mỗi nút thăm một lần: nhánh `down` chỉ đi vào cây con **không chứa** `target`, các cây con ấy rời nhau), `O(h)` bộ nhớ ngăn xếp, không cần bản đồ cha.

## 8. TRACE từng bước

**Trace Cách 2 (BFS)** với ví dụ 1, `target = 5`, `k = 2`:

| tầng | hàng đợi | mở rộng từ |
|---|---|---|
| 0 | `[5]` | — |
| 1 | `[6, 2, 3]` | con `6`, `2`; cha `3` |
| 2 | `[7, 4, 1]` | con của `2` là `7`, `4`; con của `3`: `1` (và `5` đã thăm) |

Kết quả `[7, 4, 1]` ✓. Hàm trace:

In [12]:
def trace_bfs(root, target, k):
    parent = parent_map(root)
    level, seen = [target], {target}
    print(f"tầng 0: {[n.val for n in level]}")
    for d in range(1, k + 1):
        nxt = []
        for node in level:
            for name, nb_ in (("trái", node.left), ("phải", node.right), ("cha", parent[node])):
                if nb_ is None:
                    continue
                if nb_ in seen:
                    print(f"   từ {node.val} qua {name} {nb_.val}: đã thăm, bỏ")
                    continue
                seen.add(nb_)
                nxt.append(nb_)
                print(f"   từ {node.val} qua {name} -> {nb_.val}")
        level = nxt
        print(f"tầng {d}: {[n.val for n in level]}")
        if not level:
            break
    return [n.val for n in level]


assert sorted(trace_bfs(root1, find(root1, 5), 2)) == [1, 4, 7]

tầng 0: [5]
   từ 5 qua trái -> 6
   từ 5 qua phải -> 2
   từ 5 qua cha -> 3
tầng 1: [6, 2, 3]
   từ 6 qua cha 5: đã thăm, bỏ
   từ 2 qua trái -> 7
   từ 2 qua phải -> 4
   từ 2 qua cha 5: đã thăm, bỏ
   từ 3 qua trái 5: đã thăm, bỏ
   từ 3 qua phải -> 1
tầng 2: [7, 4, 1]


**Trace Cách 4 (một lượt)**: `dfs` ghi lại giá trị trả về của từng nút.

- `dfs(3)` gọi `dfs(5)` (là `target`): `down(5, 2)` thu `7`, `4` (qua `2`); trả `0`.
- Tại `3`: `L = 0 → d = 1`, `1 < 2` nên `down(1, 2 − 1 − 1 = 0)` thu `1`; trả `1`.

Kết quả `7, 4, 1` ✓.

In [13]:
def trace_onepass(root, target, k):
    res = []
    indent = [0]

    def down(node, d):
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            print("  " * indent[0] + f"  thu {node.val}")
            return
        down(node.left, d - 1)
        down(node.right, d - 1)

    def dfs(node):
        if node is None:
            return -1
        pad = "  " * indent[0]
        indent[0] += 1
        if node is target:
            print(f"{pad}dfs({node.val}): là target, down(k={k})")
            down(node, k)
            indent[0] -= 1
            print(f"{pad}-> 0")
            return 0
        L = dfs(node.left)
        if L != -1:
            d = L + 1
            print(f"{pad}dfs({node.val}): target ở trái, d = {d}")
            if d == k:
                res.append(node.val); print(f"{pad}  thu {node.val}")
            elif d < k:
                down(node.right, k - d - 1)
            indent[0] -= 1
            return d
        R = dfs(node.right)
        if R != -1:
            d = R + 1
            print(f"{pad}dfs({node.val}): target ở phải, d = {d}")
            if d == k:
                res.append(node.val); print(f"{pad}  thu {node.val}")
            elif d < k:
                down(node.left, k - d - 1)
            indent[0] -= 1
            return d
        indent[0] -= 1
        return -1

    dfs(root)
    print("=>", res)
    return res


assert sorted(trace_onepass(root1, find(root1, 5), 2)) == [1, 4, 7]

  dfs(5): là target, down(k=2)
      thu 7
      thu 4
  -> 0
dfs(3): target ở trái, d = 1
    thu 1
=> [7, 4, 1]


## 9. Code nộp LeetCode

Chọn **Cách 4** (một lượt, không cần bản đồ cha). Giữ nguyên `class Solution`, `distanceK(self, root, target, k)`; `TreeNode` do LeetCode cung cấp.

In [14]:
class Solution:
    def distanceK(self, root: TreeNode, target: TreeNode, k: int) -> List[int]:
        res = []

        def down(node, d):                                # thu nút ở độ sâu d bên dưới node
            if node is None:
                return
            if d == 0:
                res.append(node.val)
                return
            down(node.left, d - 1)
            down(node.right, d - 1)

        def dfs(node):                                    # khoảng cách tới target, -1 nếu target không nằm dưới node
            if node is None:
                return -1
            if node is target:
                down(node, k)
                return 0
            for child, other in ((node.left, node.right), (node.right, node.left)):
                dist = dfs(child)
                if dist != -1:
                    d = dist + 1
                    if d == k:
                        res.append(node.val)
                    elif d < k:
                        down(other, k - d - 1)            # rẽ sang nhánh anh em
                    return d
            return -1

        dfs(root)
        return res


r1 = build_tree(ex1_list)
print(sorted(Solution().distanceK(r1, find(r1, 5), 2)), Solution().distanceK(ONE, ONE, 3))
r2 = build_tree([1])
assert sorted(Solution().distanceK(r1, find(r1, 5), 2)) == [1, 4, 7] and Solution().distanceK(r2, r2, 3) == []

[1, 4, 7] []


## 10. Vì sao đúng

**(a) Khoảng cách trên cây.** Giữa hai nút có đúng một đường đơn: lên tới LCA rồi xuống. Mỗi nút `v` ở khoảng cách `k` tới `target` thuộc đúng một trong hai loại: `v` nằm **dưới** `target` (độ sâu `k` trong cây con của `target`), hoặc có tổ tiên `a` của `target` (cách `target` `d ≥ 1` cạnh) sao cho `v = a` (`d = k`) hoặc `v` nằm trong cây con của `a` **không chứa** `target` ở độ sâu `k − d` (tính từ `a`, tức `k − d − 1` tính từ con kia của `a`).

**(b) Cách 2 & 3 (bản đồ cha).** Đồ thị vô hướng với cạnh `nút–con` và `nút–cha`; BFS cho khoảng cách ngắn nhất, và trên cây mọi nút ở tầng `k` đúng là nút ở khoảng cách `k`. Cách 3 không quay lại nút vừa đến là đủ để không vòng, vì cây không có chu trình và mỗi nút được tới duy nhất bằng một đường.

**(c) Cách 4.** `dfs(node)` trả `d = dist(node, target)` khi `target` thuộc cây con `node`. Với tổ tiên `a` ở cách `d`: nếu `d == k` thu `a`; nếu `d < k` thu các nút cách `a` đúng `k` mà không đi qua nhánh chứa `target`: đó là các nút ở độ sâu `k − d − 1` trong cây con kia. Nhánh chứa `target` đã được tính (nằm dưới các tổ tiên gần hơn hoặc dưới `target`). Mỗi nút được thu đúng một lần vì phân loại ở (a) rời nhau.

**(d) `k = 0`.** Chỉ `target` cách chính nó `0`: `down(target, 0)` thu `target`, và tổ tiên có `d ≥ 1 > 0` không thu gì.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **Floyd–Warshall** trên mọi hình dạng cây tới 7 nút (mục 12D) và khớp cách 1 trên cây ngẫu nhiên tới 500 nút (12E–F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đường từ gốc | `O(n·h)` | `O(n·h)` | oracle phụ |
| 2. Bản đồ cha + BFS | `O(n)` | `O(n)` | trực quan |
| 3. Bản đồ cha + DFS | `O(n)` | `O(n)` | ngắn |
| 4. Một lượt DFS | `O(n)` | `O(h)` | **đáp án chính**, không bản đồ cha |

Đo thực tế trên cây 500 nút (ngẫu nhiên và dạng chuỗi):

In [15]:
def random_tree(rng, n, chain_bias=0.0):
    """Cây n nút, giá trị duy nhất 0..500; gắn nút mới vào một chỗ trống ngẫu nhiên (chain_bias: ưu tiên chỗ mới nhất -> cây sâu)."""
    vals = rng.sample(range(501), n)
    root = TreeNode(vals[0])
    slots = [(root, "left"), (root, "right")]
    for v in vals[1:]:
        idx = len(slots) - 1 - rng.randrange(min(2, len(slots))) if rng.random() < chain_bias else rng.randrange(len(slots))
        parent, side = slots.pop(idx)
        node = TreeNode(v)
        setattr(parent, side, node)
        slots.append((node, "left"))
        slots.append((node, "right"))
    return root


def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(863)
for label, bias in [("ngẫu nhiên", 0.0), ("sâu (bias 0.95)", 0.95)]:
    root = random_tree(rng, 500, bias)
    nodes = all_nodes(root)
    target = nodes[len(nodes) // 2]
    row = [f"{name} {bench(fn, (root, target, 6)) * 1e3:7.3f} ms" for name, fn in
           [("brute", distance_k_brute), ("bfs", distance_k_bfs), ("dfs", distance_k_dfs), ("onepass", distance_k_onepass)]]
    print(f"{label:16s}: " + " | ".join(row))

ngẫu nhiên      : brute   0.565 ms | bfs   0.141 ms | dfs   0.117 ms | onepass   0.044 ms
sâu (bias 0.95) : brute   3.077 ms | bfs   0.124 ms | dfs   0.127 ms | onepass   0.043 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi hình dạng cây tới 7 nút (mọi `target`, mọi `k`) với oracle Floyd–Warshall, (E) ngẫu nhiên có seed, (F) giới hạn 500 nút, `k` tới 1000.

### A. Ví dụ của đề

In [16]:
INF = 10 ** 9


def oracle(root, target, k):
    """Floyd–Warshall trên đồ thị vô hướng của cây: trả danh sách giá trị đã sort các nút có dist == k."""
    nodes = all_nodes(root)
    idx = {id(n): i for i, n in enumerate(nodes)}
    n = len(nodes)
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for node in nodes:
        for child in (node.left, node.right):
            if child:
                a, b = idx[id(node)], idx[id(child)]
                dist[a][b] = dist[b][a] = 1
    for m in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][m] + dist[m][j] < dist[i][j]:
                    dist[i][j] = dist[i][m] + dist[m][j]
    t = idx[id(target)]
    return sorted(nodes[i].val for i in range(n) if dist[t][i] == k)


IMPLS = {
    "brute": distance_k_brute,
    "bfs": distance_k_bfs,
    "dfs": distance_k_dfs,
    "onepass": distance_k_onepass,
    "Solution": lambda r, t, k: Solution().distanceK(r, t, k),
    "SolutionBFS": lambda r, t, k: SolutionBFS().distanceK(r, t, k),
    "SolutionDFS": lambda r, t, k: SolutionDFS().distanceK(r, t, k),
    "SolutionOnePass": lambda r, t, k: SolutionOnePass().distanceK(r, t, k),
}


def check(root, target, k, want=None, names=None):
    if want is None:
        want = oracle(root, target, k)
    for name in (names or IMPLS):
        got = sorted(IMPLS[name](root, target, k))             # thứ tự không quan trọng: sort trước khi so
        assert got == want, (name, to_level(root), target.val, k, got, want)


r = build_tree(ex1_list)
check(r, find(r, 5), 2, [1, 4, 7])
r = build_tree([1])
check(r, r, 3, [])
r = build_tree(ex1_list)
assert oracle(r, find(r, 5), 2) == [1, 4, 7]
print("ví dụ đề: oracle Floyd–Warshall và", len(IMPLS), "cài đặt đúng (so sánh sau khi sort)")

ví dụ đề: oracle Floyd–Warshall và 8 cài đặt đúng (so sánh sau khi sort)


### B. Edge case

- Cây một nút, `k = 0` → `[target]`; `k ≥ 1` → `[]`.
- `k = 0` trên cây lớn → `[target.val]`.
- `target` là gốc, `target` là lá, `target` ở giữa.
- `k` lớn hơn chiều cao (kết quả rỗng).
- Cây lệch trái, lệch phải (dạng chuỗi).
- `target` là lá sâu nhất và `k` đủ để lên tới gốc rồi xuống nhánh kia.

In [17]:
r = build_tree([1])
check(r, r, 0, [1])
check(r, r, 1, [])
r = build_tree(ex1_list)
check(r, find(r, 5), 0, [5])
check(r, find(r, 3), 1, [1, 5])                          # target = gốc
check(r, find(r, 3), 3, [4, 7])
check(r, find(r, 7), 4, [1])                             # lá sâu: 7 -> 2 -> 5 -> 3 -> 1 (lên tới gốc rồi xuống nhánh kia)
check(r, find(r, 7), 5, [0, 8])
check(r, find(r, 7), 6, [])
check(r, find(r, 6), 1000, [])
left_chain = build_tree([5, 4, None, 3, None, 2, None, 1])
check(left_chain, find(left_chain, 3), 2, [1, 5])
right_chain = build_tree([1, None, 2, None, 3, None, 4])
check(right_chain, find(right_chain, 2), 1, [1, 3])
check(right_chain, find(right_chain, 4), 3, [1])
print("edge case: tất cả đúng")

edge case: tất cả đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_only_down(root, target, k):                    # chỉ đi xuống từ target
    res = []

    def down(node, d):
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            return
        down(node.left, d - 1)
        down(node.right, d - 1)

    down(target, k)
    return res


def bug_no_from(root, target, k):                      # có cha nhưng không bỏ nút vừa đến: đếm trùng
    parent = parent_map(root)
    res = []

    def dfs(node, d):
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            return
        for nxt in (node.left, node.right, parent[node]):
            dfs(nxt, d - 1)                            # SAI: quay lại cả nút vừa đến

    dfs(target, k)
    return res


def bug_k_off_by_one(root, target, k):                 # BFS dừng sớm một tầng
    return distance_k_bfs(root, target, k - 1) if k > 0 else [target.val]


def bug_only_ancestors(root, target, k):               # đi lên được nhưng không rẽ sang nhánh anh em
    parent = parent_map(root)
    res = []

    def down(node, d):
        if node is None:
            return
        if d == 0:
            res.append(node.val)
            return
        down(node.left, d - 1)
        down(node.right, d - 1)

    down(target, k)
    node, d = target, 0
    while parent[node] is not None:
        node, d = parent[node], d + 1
        if d == k:
            res.append(node.val)                       # chỉ thu bản thân tổ tiên
    return res


def bug_k0_empty(root, target, k):                     # quên k = 0 -> chính target
    return [] if k == 0 else distance_k_bfs(root, target, k)


BUGS = [bug_only_down, bug_no_from, bug_k_off_by_one, bug_only_ancestors, bug_k0_empty]
rng = random.Random(7)
cases = []
for _ in range(300):
    root = random_tree(rng, rng.randint(1, 9), 0.3)
    nodes = all_nodes(root)
    cases.append((root, rng.choice(nodes), rng.randint(0, 5)))
for bug in BUGS:
    bad = None
    for root, target, k in cases:
        try:
            got = sorted(bug(root, target, k))
        except RecursionError:
            got = "RecursionError"
        if got != oracle(root, target, k):
            bad = (root, target, k, got)
            break
    assert bad is not None, bug.__name__
    root, target, k, got = bad
    print(f"{bug.__name__:18s} cây {to_level(root)}, target {target.val}, k = {k}: ra {got}, đúng {oracle(root, target, k)}")

bug_only_down      cây [142, 490, 361, None, 118, None, 212, None, None, 194, 183, None, None, None, 349, 452], target 183, k = 3: ra [], đúng [142]
bug_no_from        cây [485, 77, 202, None, 333, 37, None, 24], target 485, k = 4: ra [37, 37, 37, 333, 333, 333, 333, 485, 485, 485, 485, 485, 485], đúng []
bug_k_off_by_one   cây [485, 77, 202, None, 333, 37, None, 24], target 485, k = 4: ra [24], đúng []
bug_only_ancestors cây [336, 479, 179, None, 79, 280, None, None, None, None, 67, 7, 10], target 280, k = 4: ra [], đúng [79]
bug_k0_empty       cây [52, None, 297, None, 292], target 292, k = 0: ra [], đúng [292]


### D. Vét cạn mọi hình dạng cây nhỏ

Mọi hình dạng cây nhị phân với `n = 1..7` nút (số Catalan: `1, 2, 5, 14, 42, 132, 429`, tổng `625` hình dạng), giá trị gán ngẫu nhiên khác nhau, **mọi** `target` và **mọi** `k` từ `0` tới `n + 1`. Oracle: Floyd–Warshall.

In [19]:
def shapes(n, memo={}):
    if n in memo:
        return memo[n]
    if n == 0:
        out = [None]
    else:
        out = [(l, r) for k in range(n) for l in shapes(k) for r in shapes(n - 1 - k)]
    memo[n] = out
    return out


def from_shape(shape, vals):
    if shape is None:
        return None
    node = TreeNode(next(vals))
    node.left = from_shape(shape[0], vals)
    node.right = from_shape(shape[1], vals)
    return node


assert [len(shapes(n)) for n in range(1, 8)] == [1, 2, 5, 14, 42, 132, 429]
t0 = time.perf_counter()
rng = random.Random(2024)
cnt = 0
for n in range(1, 8):
    for sh in shapes(n):
        vals = rng.sample(range(501), n)
        root = from_shape(sh, iter(vals))
        for target in all_nodes(root):
            for k in range(0, n + 2):
                want = oracle(root, target, k)
                for name in ("brute", "bfs", "dfs", "onepass", "Solution"):
                    assert sorted(IMPLS[name](root, target, k)) == want, (name, to_level(root), target.val, k)
                cnt += 1
print(sum(len(shapes(n)) for n in range(1, 8)), "hình dạng,", cnt, "bộ (cây, target, k): 5 cài đặt khớp Floyd–Warshall;", round(time.perf_counter() - t0, 1), "s")

625 hình dạng, 35263 bộ (cây, target, k): 5 cài đặt khớp Floyd–Warshall; 1.9 s


### E. Ngẫu nhiên có seed

Cây ngẫu nhiên tới 40 nút (ngẫu nhiên và sâu), mọi `k` tới 12, so với Floyd–Warshall; thêm tính chất: (1) các nút ở khoảng cách `k = 0, 1, 2, …` **rời nhau** và hợp lại là toàn bộ cây; (2) tổng kích thước các lớp bằng `n`.

In [20]:
rng = random.Random(20240863)
cnt = 0
for _ in range(250):
    n = rng.randint(1, 40)
    root = random_tree(rng, n, rng.choice([0.0, 0.5, 0.95]))
    target = rng.choice(all_nodes(root))
    for k in range(0, 13):
        check(root, target, k)
        cnt += 1
    layers = [set(Solution().distanceK(root, target, k)) for k in range(n + 1)]
    assert sum(len(s) for s in layers) == n and set().union(*layers) == {nd.val for nd in all_nodes(root)}
print(cnt, "bộ (cây, target, k) ngẫu nhiên: khớp Floyd–Warshall; các lớp khoảng cách chia đúng n nút")

3250 bộ (cây, target, k) ngẫu nhiên: khớp Floyd–Warshall; các lớp khoảng cách chia đúng n nút


### F. Giới hạn: 500 nút, `k` tới 1000

Cây 500 nút (ngẫu nhiên, chuỗi trái, chuỗi phải, "chữ V"): so sánh mọi cách với Cách 1 (đường từ gốc) và đối chiếu sự kiện `k ≥ n` ⇒ rỗng.

In [21]:
def chain_tree(n, side):
    vals = list(range(n))
    root = TreeNode(vals[0])
    cur = root
    for v in vals[1:]:
        nxt = TreeNode(v)
        setattr(cur, side, nxt)
        cur = nxt
    return root


rng = random.Random(500)
trees = [random_tree(rng, 500, 0.0), random_tree(rng, 500, 0.95), chain_tree(500, "left"), chain_tree(500, "right")]
cnt = 0
for root in trees:
    nodes = all_nodes(root)
    for _ in range(25):
        target = rng.choice(nodes)
        for k in (0, 1, 2, 7, 50, 250, 499, 500, 1000):
            want = sorted(distance_k_brute(root, target, k))
            for name in ("bfs", "dfs", "onepass", "Solution", "SolutionBFS", "SolutionDFS", "SolutionOnePass"):
                assert sorted(IMPLS[name](root, target, k)) == want, (name, k)
            cnt += 1
    assert Solution().distanceK(root, nodes[0], 1000) == []
print(cnt, "bộ (cây 500 nút, target, k): 7 cài đặt khớp Cách 1; k = 1000 luôn rỗng")

900 bộ (cây 500 nút, target, k): 7 cài đặt khớp Cách 1; k = 1000 luôn rỗng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ đi xuống

Quên rằng khoảng cách được tính qua cả **cha**.

In [22]:
r = build_tree(ex1_list)
print("chỉ xuống:", sorted(bug_only_down(r, find(r, 5), 2)), "| đúng:", sorted(Solution().distanceK(r, find(r, 5), 2)))
assert sorted(bug_only_down(r, find(r, 5), 2)) == [4, 7]

chỉ xuống: [4, 7] | đúng: [1, 4, 7]


**Vì sao sai:** nút `1` cách `5` hai cạnh (`5 → 3 → 1`) nhưng không nằm dưới `5`. **Đúng:** dựng bản đồ cha hoặc dùng DFS trả khoảng cách.

### Lỗi 2: không bỏ nút vừa đến

Có `parent` nhưng đệ quy mọi hàng xóm: quay lại nút vừa rời, đếm trùng hoặc đếm sai.

In [23]:
r = build_tree(ex1_list)
got = sorted(bug_no_from(r, find(r, 5), 2))
print("không bỏ nút vừa đến:", got, "| đúng:", sorted(Solution().distanceK(r, find(r, 5), 2)))
assert got != [1, 4, 7] and 5 in got

không bỏ nút vừa đến: [1, 4, 5, 5, 5, 7] | đúng: [1, 4, 7]


**Vì sao sai:** đi `5 → 2 → 5` rồi tính `5` ở khoảng cách `2`, cùng với các nút khác bị lặp. **Đúng:** truyền `frm` và bỏ qua, hoặc dùng `seen`.

### Lỗi 3: lệch `k`

Dừng sớm một tầng (hoặc muộn một tầng).

In [24]:
r = build_tree(ex1_list)
print("k-1:", sorted(bug_k_off_by_one(r, find(r, 5), 2)), "| đúng:", sorted(Solution().distanceK(r, find(r, 5), 2)))
assert sorted(bug_k_off_by_one(r, find(r, 5), 2)) == [2, 3, 6]

k-1: [2, 3, 6] | đúng: [1, 4, 7]


**Vì sao sai:** sau `k − 1` tầng ta chỉ có các nút cách `k − 1`. **Đúng:** đúng `k` lần mở rộng.

### Lỗi 4: lên được nhưng quên nhánh anh em

Đi lên chỉ thu tổ tiên, không xuống nhánh còn lại của tổ tiên đó.

In [25]:
r = build_tree(ex1_list)
print("chỉ tổ tiên:", sorted(bug_only_ancestors(r, find(r, 5), 2)), "| đúng:", sorted(Solution().distanceK(r, find(r, 5), 2)))
assert sorted(bug_only_ancestors(r, find(r, 5), 2)) == [4, 7]

chỉ tổ tiên: [4, 7] | đúng: [1, 4, 7]


**Vì sao sai:** `1` là **con** của tổ tiên `3` ở nhánh kia, không phải tổ tiên của `5`. **Đúng:** khi đi lên `d` cạnh tới `a`, thu thêm các nút ở độ sâu `k − d − 1` dưới con còn lại của `a`.

### Lỗi 5: quên `k = 0`

`k = 0` phải trả về chính `target`.

In [26]:
r = build_tree(ex1_list)
print("k = 0:", bug_k0_empty(r, find(r, 5), 0), "| đúng:", Solution().distanceK(r, find(r, 5), 0))
assert bug_k0_empty(r, find(r, 5), 0) == [] and Solution().distanceK(r, find(r, 5), 0) == [5]

k = 0: [] | đúng: [5]


**Vì sao sai:** khoảng cách của `target` tới chính nó là `0`. **Đúng:** kiểm tra `d == 0` **trước** khi mở rộng.

### Lỗi 6 (bonus): coi `target` là số nguyên

Trên LeetCode `target` là **nút**. Nếu hàm so `node is target` mà truyền `5` (số), không bao giờ khớp. Dùng `find(root, val)` để lấy nút, hoặc so `node.val == target.val`.

In [27]:
r = build_tree(ex1_list)
res = Solution().distanceK(r, 5, 2)                    # truyền SỐ thay vì NÚT
print("target là số 5 ->", res, "(không tìm thấy nút target nào)")
assert res == []

target là số 5 -> [] (không tìm thấy nút target nào)


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "khoảng cách tới một nút trong cây, kể cả đi lên" → cây là **đồ thị vô hướng**: thêm con trỏ cha (bản đồ cha) rồi BFS/DFS; hoặc để đệ quy trả khoảng cách ngược lên.

**Mẹo:**

1. Cần đi lên → dựng `parent` (khoá bằng **nút**).
2. BFS cho "đúng `k` tầng"; DFS với `frm` cho gọn.
3. Trong đệ quy trả khoảng cách, nhớ **rẽ sang nhánh anh em** với `k − d − 1`.
4. `k = 0` và `k` lớn (> độ sâu) là hai edge case kinh điển.
5. Thứ tự kết quả tuỳ ý: test bằng `sorted`.
6. Dựng cây từ list kiểu LeetCode bằng `deque`.

**Họ bài cùng pattern:** Amount of Time for Binary Tree to Be Infected (2385), Closest Leaf in a Binary Tree (742), Binary Tree Cameras, Lowest Common Ancestor of a Binary Tree (236), Diameter of Binary Tree (543), Find Distance in a Binary Tree (1740, Premium), Smallest Subtree with all the Deepest Nodes (865).

## 15. Biến thể

1. **Số nút cách đúng `k`** (chỉ đếm).
2. **Mọi nút cách ≤ `k`** (bán kính `k`).
3. **Thời gian lây lan** (2385): `max` khoảng cách từ `start` tới mọi nút (độ lệch tâm).
4. **Khoảng cách giữa hai nút** (1740) qua LCA.
5. **Đường kính** (543).

Cài biến thể 2, 3, 4 và đối chiếu Floyd–Warshall:

In [28]:
def within_k(root, target, k):
    """Mọi nút cách <= k."""
    parent = parent_map(root)
    level, seen, out = [target], {target}, [target.val]
    for _ in range(k):
        nxt = []
        for node in level:
            for nb_ in (node.left, node.right, parent[node]):
                if nb_ is not None and nb_ not in seen:
                    seen.add(nb_)
                    nxt.append(nb_)
        level = nxt
        out += [n.val for n in level]
    return sorted(out)


def eccentricity(root, target):
    """Thời gian lây lan tối thiểu (2385): số tầng BFS từ target để phủ mọi nút."""
    parent = parent_map(root)
    level, seen, d = [target], {target}, -1
    while level:
        d += 1
        nxt = []
        for node in level:
            for nb_ in (node.left, node.right, parent[node]):
                if nb_ is not None and nb_ not in seen:
                    seen.add(nb_)
                    nxt.append(nb_)
        level = nxt
    return d


def dist_two(root, a, b):
    """Khoảng cách giữa hai nút qua LCA (1740)."""
    paths = root_paths(root)
    pa, pb = paths[a], paths[b]
    c = 0
    while c < min(len(pa), len(pb)) and pa[c] is pb[c]:
        c += 1
    return len(pa) + len(pb) - 2 * c


rng = random.Random(15)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 15), 0.4)
    nodes = all_nodes(root)
    a, b = rng.choice(nodes), rng.choice(nodes)
    n = len(nodes)
    for k in range(0, 6):
        assert within_k(root, a, k) == sorted(v for kk in range(k + 1) for v in oracle(root, a, kk))
    ecc = max(kk for kk in range(n + 1) if oracle(root, a, kk))
    assert eccentricity(root, a) == ecc
    d = dist_two(root, a, b)
    assert b.val in oracle(root, a, d)
print("within_k, eccentricity (2385), dist_two (1740) khớp Floyd–Warshall trên 200 cây")

within_k, eccentricity (2385), dist_two (1740) khớp Floyd–Warshall trên 200 cây


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đường từ gốc | `|pa| + |pb| − 2·tiền tố chung` | `O(n·h)` | `O(n·h)` | oracle phụ |
| 2. Cha + BFS | tầng từ target | `O(n)` | `O(n)` | trực quan, dễ mở rộng |
| 3. Cha + DFS | bỏ nút vừa đến | `O(n)` | `O(n)` | ngắn |
| 4. Một lượt DFS | trả khoảng cách ngược lên | `O(n)` | `O(h)` | **đáp án chính** |

Cả bốn khớp oracle Floyd–Warshall trên 625 hình dạng cây nhỏ (mọi `target`, mọi `k`), 250 cây ngẫu nhiên và các cây 500 nút (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
r = build_tree(ex1_list)
print(sorted(Solution().distanceK(r, find(r, 2), 1)), sorted(Solution().distanceK(r, find(r, 6), 3)), sorted(Solution().distanceK(r, find(r, 0), 4)))

[4, 5, 7] [1, 4, 7] [2, 6]


### Bài 2 (tính tay)

Trên cây ví dụ 1, tìm mọi nút cách nút `1` đúng `3` cạnh. Vẽ đường đi (lên/xuống) cho từng nút. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện bản đồ cha + BFS:

In [30]:
def bfs_todo(root, target, k):
    parent = {}                     # TODO 1: dựng parent[nút] = cha (gốc -> None)
    level, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in level:
            pass                    # TODO 2: thêm left, right, parent[node] chưa thăm vào nxt và seen
        level = nxt
    return [n.val for n in level]


# Khi làm xong, bỏ comment để kiểm tra:
# r = build_tree(ex1_list); assert sorted(bfs_todo(r, find(r, 5), 2)) == [1, 4, 7]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **một lượt DFS**:

In [31]:
class MySolution:
    def distanceK(self, root, target, k):
        pass            # TODO


# Khi làm xong:
# r = build_tree(ex1_list); assert sorted(MySolution().distanceK(r, find(r, 5), 2)) == [1, 4, 7]

### Bài 5 (mini challenge): đếm số nút cách đúng `k` từ **mọi** nút

Viết `count_at_distance(root, k)` trả dict `{giá trị nút: số nút cách đúng k}`. Làm bằng cách gọi `distanceK` cho từng nút (`O(n²)` chấp nhận được với `n ≤ 500`) và kiểm tra tổng đối xứng: tổng các số đếm là số **cặp có thứ tự** cách `k` (chẵn khi `k > 0`).

In [32]:
def count_at_distance(root, k):
    pass            # TODO


# Khi làm xong:
# r = build_tree(ex1_list); c = count_at_distance(r, 1); assert c[3] == 2 and sum(c.values()) == 2 * 8

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: liệt kê hàng xóm (con và cha) của nút.
- Mức 2: nút `2`: con `7`, `4`, cha `5` → `[4, 5, 7]`.
- Mức 3: `6` cách `3` cạnh: `6 → 5 → 2 → 7`, `6 → 5 → 2 → 4`, `6 → 5 → 3 → 1` → `[1, 4, 7]`; `0` cách `4`: `0 → 1 → 3 → 5 → 6` và `0 → 1 → 3 → 5 → 2` → `[2, 6]`.

**Bài 2**

- Mức 1: từ `1`: cha `3`; con `0`, `8`.
- Mức 2: cách 2 chỉ có `5` (đường `1 → 3 → 5`); cách 3 là các con của `5`: `6` và `2`.
- Mức 3: `[2, 6]` (đường `1 → 3 → 5 → 6` và `1 → 3 → 5 → 2`).

**Bài 3**

- Mức 1: duyệt cây (DFS/BFS) và ghi cha của từng con.
- Mức 2: `for nb in (node.left, node.right, parent[node])` với `nb is not None and nb not in seen`.
- Mức 3: xem Cách 2.

**Bài 4**

- Mức 1: `dfs(node)` trả khoảng cách tới `target` hoặc `-1`.
- Mức 2: `down(node, d)` thu nút ở độ sâu `d` bên dưới.
- Mức 3: khi nhánh trái chứa `target` với `d = L + 1 < k`, gọi `down(node.right, k - d - 1)`.

**Bài 5**

- Mức 1: lặp `for node in all_nodes(root)`.
- Mức 2: `len(Solution().distanceK(root, node, k))`.
- Mức 3: tổng bằng số cặp có thứ tự cách `k`: với `k = 1`, bằng `2·(n − 1)` (mỗi cạnh được tính hai lần).

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
r = build_tree(ex1_list)
assert sorted(Solution().distanceK(r, find(r, 2), 1)) == [4, 5, 7]
assert sorted(Solution().distanceK(r, find(r, 6), 3)) == oracle(r, find(r, 6), 3) == [1, 4, 7]
assert sorted(Solution().distanceK(r, find(r, 0), 4)) == oracle(r, find(r, 0), 4) == [2, 6]
print("Bài 1 OK: [4,5,7], [1,4,7], [2,6]")

Bài 1 OK: [4,5,7], [1,4,7], [2,6]


### Bài 2

Từ `1` cách đúng `3`: `[2, 6]`, theo đường `1 → 3 → 5 → 6` và `1 → 3 → 5 → 2`.

In [34]:
r = build_tree(ex1_list)
assert sorted(Solution().distanceK(r, find(r, 1), 3)) == oracle(r, find(r, 1), 3) == [2, 6]
assert oracle(r, find(r, 1), 2) == [5]
print("Bài 2 OK:", oracle(r, find(r, 1), 3))

Bài 2 OK: [2, 6]


### Bài 3

In [35]:
def bfs_answer(root, target, k):
    parent = {root: None}
    st = [root]
    while st:
        node = st.pop()
        for ch in (node.left, node.right):
            if ch:
                parent[ch] = node
                st.append(ch)
    level, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in level:
            for nb_ in (node.left, node.right, parent[node]):
                if nb_ is not None and nb_ not in seen:
                    seen.add(nb_)
                    nxt.append(nb_)
        level = nxt
    return [n.val for n in level]


r = build_tree(ex1_list)
assert sorted(bfs_answer(r, find(r, 5), 2)) == [1, 4, 7]
rng = random.Random(3)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 20), 0.4)
    t = rng.choice(all_nodes(root))
    k = rng.randint(0, 8)
    assert sorted(bfs_answer(root, t, k)) == oracle(root, t, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def distanceK(self, root, target, k):
        res = []

        def down(node, d):
            if node is None:
                return
            if d == 0:
                res.append(node.val)
                return
            down(node.left, d - 1)
            down(node.right, d - 1)

        def dfs(node):
            if node is None:
                return -1
            if node is target:
                down(node, k)
                return 0
            L = dfs(node.left)
            if L != -1:
                d = L + 1
                if d == k:
                    res.append(node.val)
                elif d < k:
                    down(node.right, k - d - 1)
                return d
            R = dfs(node.right)
            if R != -1:
                d = R + 1
                if d == k:
                    res.append(node.val)
                elif d < k:
                    down(node.left, k - d - 1)
                return d
            return -1

        dfs(root)
        return res


r = build_tree(ex1_list)
assert sorted(MySolutionAnswer().distanceK(r, find(r, 5), 2)) == [1, 4, 7]
rng = random.Random(4)
for _ in range(200):
    root = random_tree(rng, rng.randint(1, 20), 0.4)
    t = rng.choice(all_nodes(root))
    k = rng.randint(0, 8)
    assert sorted(MySolutionAnswer().distanceK(root, t, k)) == oracle(root, t, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def count_at_distance_answer(root, k):
    return {node.val: len(Solution().distanceK(root, node, k)) for node in all_nodes(root)}


r = build_tree(ex1_list)
c = count_at_distance_answer(r, 1)
assert c[3] == 2 and c[5] == 3 and c[6] == 1 and sum(c.values()) == 2 * (9 - 1)
rng = random.Random(5)
for _ in range(100):
    root = random_tree(rng, rng.randint(1, 15), 0.4)
    n = len(all_nodes(root))
    cc = count_at_distance_answer(root, 1)
    assert sum(cc.values()) == 2 * (n - 1)
    for k in range(0, 5):
        assert sum(count_at_distance_answer(root, k).values()) % 2 == (n if k == 0 else 0) % 2
print("Bài 5 OK: tổng bậc = 2(n - 1); số cặp có thứ tự cách k là chẵn khi k > 0")

Bài 5 OK: tổng bậc = 2(n - 1); số cặp có thứ tự cách k là chẵn khi k > 0


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (khoảng cách trong cây, kể cả lên cha → một lượt DFS trả khoảng cách):**

```python
class Solution:
    def distanceK(self, root, target, k):
        res = []

        def down(node, d):                       # thu nút ở độ sâu d bên dưới node
            if node is None:
                return
            if d == 0:
                res.append(node.val); return
            down(node.left, d - 1); down(node.right, d - 1)

        def dfs(node):                           # khoảng cách tới target, -1 nếu không nằm dưới node
            if node is None:
                return -1
            if node is target:
                down(node, k); return 0
            for child, other in ((node.left, node.right), (node.right, node.left)):
                dist = dfs(child)
                if dist != -1:
                    d = dist + 1
                    if d == k: res.append(node.val)
                    elif d < k: down(other, k - d - 1)
                    return d
            return -1

        dfs(root)
        return res
```

**Checklist trước khi submit:**

- [ ] Đi cả **lên** (cha hoặc DFS trả khoảng cách) lẫn xuống?
- [ ] Không quay lại nút vừa đến (`frm`/`seen`)?
- [ ] Khi đi lên `d` cạnh: thu tổ tiên nếu `d == k`, hoặc xuống nhánh **kia** với `k − d − 1`?
- [ ] `k = 0` → `[target.val]`; `k` quá lớn → `[]`?
- [ ] `target` là **nút** (so `is`), khoá `dict`/`set` bằng nút?
- [ ] Đã thử ví dụ 1 (`[7,4,1]`), ví dụ 2 (`[]`), cây lệch, chú ý độ sâu đệ quy ≤ 500?
- [ ] Đúng tên `class Solution`, `distanceK(self, root, target, k)`, không in debug.